In [65]:
import json


class ContentAgent:

    def __init__(self, llm):
        self.llm = llm

    def generate_llm_content(self, mission, patient):

        # ---------------------------------------------------------
        # 1. Extract deterministic source-of-truth data
        # ---------------------------------------------------------

        routine = mission["routine"]
        objective = mission["objective"]
        difficulty = mission["difficulty"]

        # ---------------------------------------------------------
        # 2. Build VERIFIED FACTS
        #
        # The LLM is allowed to use ONLY these facts.
        # ---------------------------------------------------------

        verified_facts = {
            "patient_name": patient["name"],
            "npc_name": mission["npc"]["name"],
            "routine": routine["title"],
            "scheduled_time": routine["scheduled_time"],
            "target_object": objective["target_object"],
            "target_action": objective["target_action"],
            "objective": objective["description"]
        }

        # ---------------------------------------------------------
        # 3. System prompt
        # ---------------------------------------------------------

        system_prompt = (
            "You are an AI content generator for an elderly "
            "cognitive training system.\n\n"

            "Your job is to generate short, warm and simple "
            "dialogue for a cognitive activity.\n\n"

            "IMPORTANT GROUNDING RULE:\n"
            "You may ONLY use facts explicitly present in "
            "VERIFIED FACTS provided by the user.\n\n"

            "Do NOT invent, assume, infer, or add facts.\n\n"

            "NEVER invent:\n"
            "- colors\n"
            "- sizes\n"
            "- shapes\n"
            "- physical appearance\n"
            "- temperature\n"
            "- locations\n"
            "- additional objects\n"
            "- ownership\n"
            "- preferences\n"
            "- memories\n"
            "- family information\n"
            "- medical information\n"
            "- health benefits\n"
            "- environmental conditions\n"
            "- weather\n\n"

            "For example, if the verified target object is "
            "'water_bottle', you may refer to it as "
            "'water bottle'.\n"
            "You must NOT describe it as a blue bottle, "
            "clear bottle, big bottle, cold bottle, or any "
            "other description unless that fact is explicitly "
            "provided in VERIFIED FACTS.\n\n"

            "COGNITIVE TRAINING RULES:\n"
            "1. Encourage memory recall.\n"
            "2. Do not immediately reveal the answer in the opening.\n"
            "3. Keep sentences short and easy to understand.\n"
            "4. Use warm and respectful language.\n"
            "5. Never mention dementia or cognitive impairment.\n"
            "6. Never use frightening, judgmental or childish language.\n"
            "7. Do not give medical advice.\n"
            "8. Do not claim that a real-world action happened "
            "unless the system explicitly confirms it.\n\n"

            "DIFFICULTY RULES:\n"
            "- very_easy: provide a gentle contextual cue.\n"
            "- easy: provide a moderate cue.\n"
            "- medium: provide only a small cue.\n"
            "- hard: encourage independent recall.\n\n"

            "SUCCESS MESSAGE RULE:\n"
            "The success message must be a generic encouraging message.\n"
            "Do NOT claim that the patient remembered, completed, "
            "performed, or successfully carried out a specific task.\n"
            "Do NOT claim that a real-world action occurred.\n"
            "Do NOT use words such as 'perfectly', 'correctly', "
            "'successfully', or 'on time' unless those facts are "
            "explicitly provided as verified input.\n"
        )

        # ---------------------------------------------------------
        # 4. User prompt containing verified facts
        # ---------------------------------------------------------

        user_prompt = (
            "VERIFIED FACTS:\n"
            f"{json.dumps(verified_facts, ensure_ascii=False, indent=2)}\n\n"

            f"CURRENT DIFFICULTY:\n"
            f"{difficulty}\n\n"

            "Generate exactly three pieces of content:\n"
            "1. opening dialogue\n"
            "2. one gentle hint\n"
            "3. one success message\n\n"

            "The opening should encourage the patient to remember "
            "the routine.\n\n"

            "The hint should help recall without unnecessarily "
            "revealing the answer.\n\n"

            "The success message should encourage the patient "
            "after successful recall.\n\n"

            "Use ONLY information from VERIFIED FACTS."
        )

        # ---------------------------------------------------------
        # 5. Strict JSON schema
        # ---------------------------------------------------------

        response_format = {
            "type": "json_schema",
            "json_schema": {
                "name": "cognitive_game_content",
                "strict": True,
                "schema": {
                    "type": "object",

                    "properties": {
                        "opening": {
                            "type": "string"
                        },

                        "hint": {
                            "type": "string"
                        },

                        "success": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "opening",
                        "hint",
                        "success"
                    ],

                    "additionalProperties": False
                }
            }
        }

        # ---------------------------------------------------------
        # 6. Call LLM
        # ---------------------------------------------------------

        response = self.llm.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            max_completion_tokens=150,
            response_format=response_format
        )

        # ---------------------------------------------------------
        # 7. Parse structured JSON response
        # ---------------------------------------------------------

        content = json.loads(response["text"])

        return content


print("ContentAgent class loaded successfully.")

ContentAgent class loaded successfully.


In [66]:
mission = {
    "mission_id": "mission_water_1030",
    "mission_type": "routine_recall",
    "difficulty": "very_easy",

    "routine": {
        "routine_id": "water_1030",
        "title": "Drink water",
        "scheduled_time": "10:30"
    },

    "world": {
        "location": "home",
        "starting_location": "living_room",
        "target_location": "kitchen"
    },

    "npc": {
        "name": "Anu",
        "role": "family_member",
        "opening_dialogue":
            "Amma, I think we have something to do."
    },

    "objective": {
        "description":
            "Remember what you usually do around this time.",
        "target_object": "water_bottle",
        "target_action": "drink_water"
    },

    "steps": [
        "talk_to_npc",
        "remember_routine",
        "walk_to_kitchen",
        "find_water_bottle",
        "identify_object",
        "perform_action"
    ],

    "cognitive_targets": [
        "routine_recall",
        "time_orientation",
        "attention",
        "object_recognition",
        "action_recall"
    ],

    "completion": {
        "real_world_action_required": True,
        "cognitive_response_required": True
    }
}

print("Test mission created!")

Test mission created!


In [67]:
content_agent = ContentAgent(
    llm=llm
)

print("Content Agent ready!")

Content Agent ready!


In [68]:
patient = {
    "patient_id": "lakshmi_001",
    "name": "Lakshmi",
    "preferred_language": "te",
    "supported_languages": ["te", "en", "hi", "as"],
    "interaction_mode": "voice_touch",
    "voice_enabled": True,
    "voice_input_enabled": True,
    "voice_output_enabled": True,
    "status": "active"
}

print("Patient data ready!")

Patient data ready!


In [69]:
verified_facts = {
    "patient_name": patient["name"],
    "npc_name": mission["npc"]["name"],
    "routine": mission["routine"]["title"],
    "scheduled_time": mission["routine"]["scheduled_time"],
    "target_object": mission["objective"]["target_object"],
    "target_action": mission["objective"]["target_action"],
    "objective": mission["objective"]["description"]
}

print("Verified facts:")
print(json.dumps(
    verified_facts,
    indent=2,
    ensure_ascii=False
))

Verified facts:
{
  "patient_name": "Lakshmi",
  "npc_name": "Anu",
  "routine": "Drink water",
  "scheduled_time": "10:30",
  "target_object": "water_bottle",
  "target_action": "drink_water",
  "objective": "Remember what you usually do around this time."
}


In [70]:
llm_content = content_agent.generate_llm_content(mission, patient)

print(json.dumps(
    llm_content,
    indent=2,
    ensure_ascii=False
))

{
  "opening": "Hello Lakshmi, it’s Anu. I was thinking about what we usually do together around this time.",
  "hint": "Remember the routine you have at 10:30 every day.",
  "success": "Great job recalling what comes next!"
}


In [6]:
npc_content = content_agent.generate_npc_dialogue(
    mission
)

print(npc_content)

{'npc_name': 'Anu', 'opening': 'Amma, I think we have something to do.', 'hint': 'Can you remember what we usually do around this time?', 'object_prompt': 'Can you find the water bottle?', 'success': 'Well done! You remembered what to do.'}


In [7]:
activity_content = content_agent.generate_activity_text(
    mission
)

print(activity_content)

{'title': 'Daily Routine Mission', 'instruction': 'Remember what you usually do around this time.', 'goal': 'Find the water bottle and complete the activity.', 'steps': ['talk_to_npc', 'remember_routine', 'walk_to_kitchen', 'find_water_bottle', 'identify_object', 'perform_action']}


In [1]:
!pip install groq

In [3]:
!pip install python-dotenv

In [3]:
from groq import Groq

groq_client = Groq()

print("Groq client ready.")

Groq client ready.


In [4]:
from groq import Groq

groq_client = Groq()

print("Groq client ready.")

Groq client ready.


In [7]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ],
    max_completion_tokens=100,
    include_reasoning=False,
    stream=False
)

print("FULL RESPONSE:")
print(response)

print("\nCONTENT:")
print(response.choices[0].message.content)

print("\nUSAGE:")
print(response.usage)

FULL RESPONSE:
ChatCompletion(id='chatcmpl-97f18812-d73e-4931-9d3f-1c7002507f42', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1789180458, model='openai/gpt-oss-20b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_d3e146e1a5', usage=CompletionUsage(completion_tokens=48, prompt_tokens=78, total_tokens=126, completion_time=0.053454927, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=37), prompt_time=0.004252038, prompt_tokens_details=None, queue_time=0.314698151, total_time=0.057706965), usage_breakdown=None, x_groq=XGroq(id='req_01m29qeddve9aag4k9nmtces8h', debug=None, seed=1934034930, usage=None))

CONTENT:
Hello!

USAGE:
CompletionUsage(completion_tokens=48, prompt_tokens=78, total_tokens=126, completion_time=0.053454927, completion_to

In [8]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ],
    max_completion_tokens=100,
    include_reasoning=False
)

print(response.choices[0].message.content)

Hello! How can I help you today?


In [9]:
class LLMUsageGuard:

    def __init__(
        self,
        max_calls=20,
        max_completion_tokens=4000
    ):
        self.max_calls = max_calls
        self.max_completion_tokens = max_completion_tokens

        self.calls_used = 0
        self.prompt_tokens_used = 0
        self.completion_tokens_used = 0
        self.total_tokens_used = 0

    def can_call(self):
        return (
            self.calls_used < self.max_calls
            and
            self.completion_tokens_used < self.max_completion_tokens
        )

    def record(self, usage):

        self.calls_used += 1

        if usage is not None:

            self.prompt_tokens_used += usage.prompt_tokens
            self.completion_tokens_used += usage.completion_tokens
            self.total_tokens_used += usage.total_tokens

    def remaining_completion_tokens(self):

        return max(
            0,
            self.max_completion_tokens
            - self.completion_tokens_used
        )

    def summary(self):

        return {
            "calls_used": self.calls_used,
            "max_calls": self.max_calls,
            "prompt_tokens_used": self.prompt_tokens_used,
            "completion_tokens_used": self.completion_tokens_used,
            "max_completion_tokens": self.max_completion_tokens,
            "total_tokens_used": self.total_tokens_used
        }

In [10]:
usage_guard = LLMUsageGuard(
    max_calls=20,
    max_completion_tokens=4000
)

print(usage_guard.summary())

{'calls_used': 0, 'max_calls': 20, 'prompt_tokens_used': 0, 'completion_tokens_used': 0, 'max_completion_tokens': 4000, 'total_tokens_used': 0}


In [28]:
class GroqLLM:

    def __init__(
        self,
        client,
        usage_guard,
        model="openai/gpt-oss-20b"
    ):
        self.client = client
        self.usage_guard = usage_guard
        self.model = model

    def generate(
        self,
        system_prompt,
        user_prompt,
        max_completion_tokens=200,
        response_format=None
    ):

        if not self.usage_guard.can_call():
            raise RuntimeError(
                "LLM usage budget exhausted."
            )

        remaining = (
            self.usage_guard.remaining_completion_tokens()
        )

        allowed_tokens = min(
            max_completion_tokens,
            remaining
        )

        if allowed_tokens <= 0:
            raise RuntimeError(
                "No completion tokens remaining."
            )

        request = {
            "model": self.model,

            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            "max_completion_tokens": allowed_tokens,

            "reasoning_effort": "low",

            "include_reasoning": False,

            "stream": False
        }

        if response_format is not None:
            request["response_format"] = response_format

        response = self.client.chat.completions.create(
            **request
        )

        self.usage_guard.record(
            response.usage
        )

        return {
            "text": response.choices[0].message.content,
            "usage": {
                "prompt_tokens":
                    response.usage.prompt_tokens,

                "completion_tokens":
                    response.usage.completion_tokens,

                "total_tokens":
                    response.usage.total_tokens
            }
        }

In [29]:
llm = GroqLLM(
    client=groq_client,
    usage_guard=usage_guard
)

print("GroqLLM ready!")

GroqLLM ready!


In [17]:
result = llm.generate(
    system_prompt=(
        "You are a friendly cognitive assistant "
        "for an elderly patient. "
        "Keep responses short and simple."
    ),
    user_prompt=(
        "Say hello to Lakshmi in one short sentence."
    ),
    max_completion_tokens=100
)

print("Response:")
print(result["text"])

print("\nUsage:")
print(result["usage"])

Response:
Hello, Lakshmi!

Usage:
{'prompt_tokens': 101, 'completion_tokens': 22, 'total_tokens': 123}


In [18]:
print(usage_guard.summary())

{'calls_used': 2, 'max_calls': 20, 'prompt_tokens_used': 202, 'completion_tokens_used': 81, 'max_completion_tokens': 4000, 'total_tokens_used': 283}


In [19]:
test_result = llm.generate(
    system_prompt=(
        "You are the content-generation component of a cognitive "
        "assistance system for an elderly patient. "
        "Generate only simple, warm, encouraging dialogue. "
        "Never invent personal memories, family members, places, "
        "medical information, or routines."
    ),
    user_prompt=(
        "Patient name: Lakshmi\n"
        "Goal: routine recall\n"
        "Current routine: Drink water\n"
        "Scheduled time: 10:30\n"
        "Difficulty: very_easy\n\n"
        "Generate one short NPC dialogue line encouraging "
        "the patient to remember the routine."
    ),
    max_completion_tokens=100
)

print(test_result["text"])
print()
print(test_result["usage"])

Good morning, Lakshmi! It’s time for your water—let's enjoy that healthy sip together at 10:30. 🌟

{'prompt_tokens': 159, 'completion_tokens': 43, 'total_tokens': 202}


In [20]:
print(usage_guard.summary())

{'calls_used': 3, 'max_calls': 20, 'prompt_tokens_used': 361, 'completion_tokens_used': 124, 'max_completion_tokens': 4000, 'total_tokens_used': 485}


In [21]:
import json

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": (
                "You generate safe cognitive-game content for elderly patients. "
                "Use only the verified facts provided. "
                "Never invent memories, family members, places, medicines, "
                "medical advice, or routines. "
                "Keep language short, warm, and easy to understand. "
                "The goal is to encourage recall, not simply reveal the answer."
            )
        },
        {
            "role": "user",
            "content": (
                "Patient: Lakshmi\n"
                "Goal: routine recall\n"
                "Routine: Drink water\n"
                "Scheduled time: 10:30\n"
                "Difficulty: very_easy\n\n"
                "Generate an opening dialogue, one gentle hint, "
                "and one success message."
            )
        }
    ],
    max_completion_tokens=150,
    reasoning_effort="low",
    include_reasoning=False,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "cognitive_game_content",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "opening": {
                        "type": "string"
                    },
                    "hint": {
                        "type": "string"
                    },
                    "success": {
                        "type": "string"
                    }
                },
                "required": [
                    "opening",
                    "hint",
                    "success"
                ],
                "additionalProperties": False
            }
        }
    }
)

content = json.loads(
    response.choices[0].message.content
)

print(json.dumps(content, indent=2, ensure_ascii=False))
print("\nUsage:")
print(response.usage)

{
  "opening": "Good morning, Lakshmi! Let’s start your day with a refreshing routine.",
  "hint": "It’s the cool drink that helps you stay hydrated and feel good.",
  "success": "Great job! You’ve remembered to drink water at 10:30. Keep it up!"
}

Usage:
CompletionUsage(completion_tokens=80, prompt_tokens=232, total_tokens=312, completion_time=0.090692214, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=7), prompt_time=0.07865709, prompt_tokens_details=None, queue_time=0.348616717, total_time=0.169349304)
